In [ ]:
!pip install fastapi streamlit langchain sentence-transformers pandas pyjwt

In [ ]:
!git clone https://github.com/springboardmentor441p-coderr/Fintech-data

In [ ]:
!ls Fintech-data

In [ ]:
import pandas as pd
df=pd.read_csv("Fintech-data/HR/hr_data.csv")
df.head(10)

In [ ]:
import os
import pandas as pd

In [ ]:
def clean_text(text):
    # Remove special characters
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = text.strip()
    # Normalize multiple spaces
    text = " ".join(text.split())
    return text

In [ ]:
folder = "Fintech-data"
for file in os.listdir(folder):
    if file.endswith(".md"):
        with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
            raw_text = f.read()
            cleaned = clean_text(raw_text)
            print(f"--- {file} ---")
            print(cleaned[:300])  # show first 300 chars

In [ ]:
for file in os.listdir(folder):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder, file))
        # Apply cleaning to text columns
        for col in df.columns:
            if df[col].dtype == "object":
                df[col] = df[col].apply(lambda x: clean_text(str(x)))
        print(f"--- {file} ---")
        print(df.head())

In [ ]:
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import word_tokenize

def chunk_text(text, chunk_size=300):
    words = word_tokenize(text)
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

In [ ]:
folder = "Fintech-data"
for file in os.listdir(folder):
    if file.endswith(".md"):
        with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
            raw_text = f.read()
            chunks = chunk_text(raw_text, chunk_size=300)
            print(f"--- {file} ---")
            print(f"Total chunks: {len(chunks)}")
            print("Sample chunk:\n", chunks[0][:200])

In [ ]:
for file in os.listdir(folder):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder, file))
        for col in df.columns:
            if df[col].dtype == "object":
                text = " ".join(df[col].dropna().astype(str).tolist())
                chunks = chunk_text(text, chunk_size=300)
                print(f"--- {file} ---")
                print(f"Total chunks: {len(chunks)}")
                print("Sample chunk:\n", chunks[0][:200])

In [ ]:
role_mapping = {
    "finance": ["finance_report.md", "quarterly_summary.csv"],
    "marketing": ["marketing_analysis.md", "campaign_data.csv"],
    "hr": ["employee_data.csv", "hr_policies.md"],
    "engineering": ["tech_docs.md", "system_architecture.md"],
    "employees": ["general_handbook.md"],
    "c_level": "all"  # C-Level can access everything
}

In [ ]:
def assign_metadata(file_name, chunk, role_mapping):
    metadata = {}
    metadata["source_file"] = file_name
    metadata["chunk_text"] = chunk
    metadata["roles"] = []

    for role, files in role_mapping.items():
        if files == "all" or file_name in files:
            metadata["roles"].append(role)
    return metadata

In [ ]:
!pip install chromadb sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
client = chromadb.Client()
collection = client.create_collection(name="company_docs", get_or_create=True)

In [ ]:
all_chunks_with_metadata = []

# Helper function to get all files in a directory and its subdirectories
def get_all_files(directory):
    file_list = []
    for root, _, files in os.walk(directory):
        for file in files:
            file_list.append(os.path.join(root, file))
    return file_list

# Get all files in the Fintech-data folder and its subfolders
all_data_files = get_all_files(folder)

# Process markdown files
for file_path in all_data_files:
    file_name = os.path.basename(file_path)
    if file_name.endswith(".md"):
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()
            cleaned_text = clean_text(raw_text)
            chunks = chunk_text(cleaned_text, chunk_size=300)
            for chunk in chunks:
                metadata = assign_metadata(file_name, chunk, role_mapping)
                all_chunks_with_metadata.append(metadata)

# Process CSV files
for file_path in all_data_files:
    file_name = os.path.basename(file_path)
    if file_name.endswith(".csv"):
        df = pd.read_csv(file_path)
        combined_text = []
        for col in df.columns:
            if df[col].dtype == "object":
                combined_text.extend(df[col].dropna().astype(str).tolist())

        if combined_text:
            text_to_chunk = " ".join(combined_text)
            chunks = chunk_text(text_to_chunk, chunk_size=300)
            for chunk in chunks:
                metadata = assign_metadata(file_name, chunk, role_mapping)
                all_chunks_with_metadata.append(metadata)

print(f"Total chunks created: {len(all_chunks_with_metadata)}")

Now that `all_chunks_with_metadata` is populated, you can re-run the cell to add the documents to your ChromaDB collection.

In [ ]:
for i, item in enumerate(all_chunks_with_metadata):
    text = item["chunk_text"]
    roles = item["roles"]
    source = item["source_file"]

    embedding = model.encode(text).tolist()

    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[embedding],
        documents=[text],
        metadatas=[{"roles": roles, "source": source}]
    )

In [ ]:
results = collection.query(
    query_texts=["latest quarterly spend"],
    n_results=3
)
print(results)

In [ ]:
def role_based_search(query, user_role, top_k=3):
    # Run semantic search
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )

    # Filter results by role
    filtered_docs = []
    for i, doc in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        if user_role in meta["roles"] or "c_level" in meta["roles"]:
            filtered_docs.append({
                "document": doc,
                "source": meta["source"],
                "roles": meta["roles"]
            })
    return filtered_docs

In [ ]:
if all_chunks_with_metadata:
    print(all_chunks_with_metadata[0])
else:
    print("The list all_chunks_with_metadata is empty.")

In [ ]:
for item in all_chunks_with_metadata[:5]:
    print(item["source_file"], item["roles"])

for i, item in enumerate(all_chunks_with_metadata):
    text = item["chunk_text"]
    roles = item["roles"]
    source = item["source_file"]

    embedding = model.encode(text).tolist()

    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[embedding],
        documents=[text],
        metadatas=[{"roles": roles, "source": source}]
    )

In [ ]:
def role_based_search(query, user_role, top_k=3):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    filtered_docs = []
    for i, doc in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        if user_role in meta["roles"] or "c_level" in meta["roles"]:
            filtered_docs.append({
                "document": doc,
                "source": meta["source"],
                "roles": meta["roles"]
            })
    return filtered_docs

finance_results = role_based_search("latest quarterly spend", "finance")
print(finance_results)